# SMF Grant Advisory Assistant — Live Demo

### RAG & Agentic AI · Mini-Project · Notebook 2 of 2 · Trial 3

**Team:** Ng Kim Chuan (2570464) · Ho Yee Hong (2570456) ·
Ivan Ong Yan Wen (2570468) · Tan Teck Guan (2570471)

---

## Why this is a separate notebook

Notebook 1 takes 10–16 minutes on a cold runtime: OCR over 8 rasterised pages,
Whisper over 4 MP3s, and Whisper again over 2 video audio tracks. Fine for a
build, impossible inside a 10-minute presentation. This notebook loads the
artifacts notebook 1 exported and is interactive in under a minute.

**Run notebook 1 first**, at least through section 12 (Export artifacts).

## The three tabs

| Tab | Who | What it shows |
|---|---|---|
| **1 · Member** | A manufacturer asking about their own situation | Grounded answers with citations, and a role guardrail: questions about other member companies are refused |
| **2 · Admin** | An SMF advisory officer | The same engine unrestricted, so the member case studies become available |
| **3 · Compare** | Baseline vs best combined | Side by side on the same question, deterministic and reproducible |

Tabs 1 and 2 run the configuration the ablation recommends **deploying** — hybrid
retrieval and reranking, with the abstention floor left at 0, so abstention comes
from the grounding contract rather than a cosine cut-off. Tab 3 has a different
job: it has to reproduce the measured experiment, so its improved side keeps the
tuned floor from experiment 4. One tab is the product, the other is the evidence.

## Every answer shows its evidence

For each source: document title, exact locator, modality badge, similarity
score, the retrieved excerpt, and a link that opens the source at that point —
`#page=N` for PDFs, `#t=SS` for audio and video. Each answer is also labelled
with its question category and the modalities the evidence came from.

Generation runs on **Groq** (`llama-3.1-8b-instant`). The pipeline's *evaluation*
ran on a small local model so the numbers are reproducible and deterministic; the
*demo* uses a larger hosted model because stronger instruction-following makes for
a better live session. Measure on the small model, present on the large one.

Without a key the demo falls back to a mock and retrieval, citations, confidence,
links and the role guardrail all still work.

Each citation carries a **confidence score**: the cosine similarity between the
question and that chunk, shown as a percentage with a High / Medium / Low band.
It is a retrieval confidence, not a calibrated probability that the answer is
correct, and the interface says so.

---
## 1. Setup

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = "/content/drive/MyDrive/Trial_3"   # EDIT to match your Drive
else:
    PROJECT_ROOT = ".."

print("Colab:", IN_COLAB, "| PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
import importlib.util, subprocess, sys

PACKAGE_IMPORTS = {
    "numpy": "numpy", "pandas": "pandas",
    "sentence-transformers": "sentence_transformers",
    "faiss-cpu": "faiss", "rank-bm25": "rank_bm25",
    "gradio": "gradio", "groq": "groq",
}

def can_import(name):
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ModuleNotFoundError, ValueError):
        return False

missing = [p for p, m in PACKAGE_IMPORTS.items() if not can_import(m)]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "--disable-pip-version-check", "-q", *missing], check=True)
    importlib.invalidate_caches()
else:
    print("All packages present.")

In [ ]:
# --- Load the artifacts notebook 1 exported -----------------------------
import json, pickle, re
from pathlib import Path
import numpy as np
import pandas as pd
import faiss

ROOT = Path(PROJECT_ROOT).resolve()
ARTIFACTS = ROOT / "artifacts"
EVAL_DIR = ROOT / "eval"
RESULTS = ROOT / "results"

required = ["faiss_index.bin", "chunks.pkl", "config.json", "chunk_embeddings.npy"]
absent = [f for f in required if not (ARTIFACTS / f).exists()]
if absent:
    raise FileNotFoundError(
        f"Missing artifacts: {absent}\n\n"
        f"Run 01_Build_and_Evaluate.ipynb through section 12 first."
    )

CONFIG = json.loads((ARTIFACTS / "config.json").read_text())
if str(CONFIG.get("artifact_source", "")).startswith("SANDBOX"):
    raise RuntimeError(
        "These artifacts came from tools/verify_notebook_01.py, which uses a "
        "STUB embedder.\nRe-run 01_Build_and_Evaluate.ipynb before demonstrating."
    )

with open(ARTIFACTS / "chunks.pkl", "rb") as fh:
    chunks = pickle.load(fh)
faiss_index = faiss.read_index(str(ARTIFACTS / "faiss_index.bin"))
CHUNK_EMB = np.load(ARTIFACTS / "chunk_embeddings.npy")
assert CHUNK_EMB.shape[0] == len(chunks), "artifacts out of sync; re-run section 12"

TOP_K         = CONFIG["top_k"]
BEST_ALPHA    = CONFIG.get("best_alpha", 0.8)
REFUSAL_TEXT  = CONFIG["refusal_text"]
SYSTEM_RULES  = CONFIG["system_rules"]

# The demo runs the configuration the ablation told us to DEPLOY, which is not
# the same as the configuration that maximised any single metric.
#
# config.json records best_threshold = 0.40, the cosine floor that maximised
# precision in experiment 3b. Section 9 of the report is explicit that we would
# not ship it: at 0.40 the floor discards weak-but-relevant evidence and hit
# rate falls 0.966 -> 0.828. In the demo the effect is worse than a number,
# because the cross-modal question - the one that proves four modalities feed
# one answer - retrieves NOTHING at 0.40 (V03: 5 chunks spanning all four
# modalities at no floor, 0 chunks at 0.40) and the assistant refuses a question
# it can in fact answer.
#
# So the demo floor is 0.0 and abstention comes from where it belongs: rule 1 of
# the grounding contract, which forbids answering from outside the retrieved
# passages. The Admin tab exposes the floor as a slider, so raising it to 0.40
# live is the fastest way to show the trade-off we measured.
DEMO_THRESHOLD = 0.0
TUNED_THRESHOLD = CONFIG.get("best_threshold", 0.4)
SIM_THRESHOLD = DEMO_THRESHOLD

# The demo pins its own model. config.json records the model the EVALUATION used
# (a small local one, for reproducibility); the demo deliberately runs a larger
# hosted model, so it must not inherit that value.
LLM_MODEL = "llama-3.1-8b-instant"
MAX_OUTPUT_TOKENS = 700

print(f"{len(chunks)} chunks | {faiss_index.ntotal} vectors")
print(f"modalities: {CONFIG['modalities']}")
print(f"retrieval: top_k={TOP_K}, alpha={BEST_ALPHA}, floor={SIM_THRESHOLD}"
      f"  (ablation-tuned floor {TUNED_THRESHOLD} available on the Admin slider)")
print(f"generation: {LLM_MODEL} (Groq)")
print("\nchunks per modality:")
for m, n in pd.Series([c["modality"] for c in chunks]).value_counts().items():
    print(f"  {m:<8} {n:>4}")

In [ ]:
# --- Retrievers ---------------------------------------------------------
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

print(f"Loading {CONFIG['embedding_model']} ...")
embedder = SentenceTransformer(CONFIG["embedding_model"])

def embed(texts):
    v = embedder.encode(texts, convert_to_numpy=True,
                        normalize_embeddings=True, show_progress_bar=False)
    return np.asarray(v, dtype=np.float32)

def tokenize(t):
    return re.findall(r"[a-z0-9$][a-z0-9$,.%]*", t.lower())

bm25_index = BM25Okapi([tokenize(c["text"]) for c in chunks])
print("Retrievers ready.")

In [ ]:
# --- Retrieval: identical logic to notebook 1 ---------------------------
# Cosine is computed against ALL chunks so the fusion is a real fusion and the
# threshold applies to every candidate, not just the FAISS top-k.
def retrieve(question, top_k=TOP_K, use_hybrid=True, alpha=BEST_ALPHA,
             sim_threshold=SIM_THRESHOLD, exclude_sources=()):
    n = len(chunks)
    pool = min(max(top_k * 4, 20), n)
    qv = embed([question])

    if use_hybrid:
        d_all = (CHUNK_EMB @ qv[0]).astype(np.float32)
        bm = np.asarray(bm25_index.get_scores(tokenize(question)), dtype=np.float32)

        def norm(a):
            lo, hi = float(a.min()), float(a.max())
            return np.zeros_like(a) if hi - lo < 1e-9 else (a - lo) / (hi - lo)

        fused = alpha * norm(d_all) + (1 - alpha) * norm(bm)
        order = np.argsort(-fused)[:pool]
        cand = [(int(i), float(fused[i]), float(d_all[i])) for i in order]
    else:
        ds, di = faiss_index.search(qv, pool)
        cand = [(int(i), float(s), float(s))
                for s, i in zip(ds[0], di[0]) if i >= 0]

    if exclude_sources:
        cand = [c for c in cand
                if not any(x in chunks[c[0]]["source"] for x in exclude_sources)]

    out = []
    for i, score, dense in cand:
        if sim_threshold > 0 and dense < sim_threshold:
            continue
        c = chunks[i]
        out.append({"rank": len(out) + 1, "chunk_id": c["chunk_id"],
                    "score": score, "dense_score": dense, "text": c["text"],
                    "title": c["title"], "source": c["source"],
                    "page": c["page"], "modality": c["modality"],
                    "extraction": c["extraction"],
                    "locator": c.get("locator", ""),
                    "t_start": c.get("t_start"), "rel_path": c.get("rel_path", "")})
        if len(out) >= top_k:
            break
    return out

---
## 2. Role policy

SMF is a federation of roughly 5,000 often-competing manufacturers. A shared
assistant that tells member A what member B claimed, for how much, and whether
they were rejected is a confidentiality incident rather than a feature. The
control has to exist regardless of what the records contain, because it is the
first thing SMF would ask about.

It is enforced at **two independent layers**, because a prompt instruction alone
is not a control — it can be argued around:

1. **Retrieval.** In the member role the four member-record sources are removed
   from the candidate pool, so the evidence never reaches the model.
2. **Intent.** A named-entity check refuses questions naming another member
   before retrieval runs, and says what it *can* help with instead.

General scheme questions are unaffected. The restriction is scoped to other
companies' records, not to the corpus.

In [ ]:
# --- Role policy --------------------------------------------------------
MEMBER_RECORD_SOURCES = (
    "KB12_SMF_Member_Case_Studies_SYNTHETIC.pdf",   # text case studies
    "KBV02_SMF_Member_Case_Study_Cards.pdf",        # image case-study cards
    "VID001_case_study_novara.mp4",                 # video testimonial
    "VID002_case_study_tanjung.mp4",                # video testimonial
)

# Company names anywhere in the corpus. ASR spellings are included because
# Whisper writes "Tenjong" for Tanjung and "Navara" for Novara - a guardrail
# that only matches the correct spelling is trivially bypassed by using the
# transcript's spelling instead.
MEMBER_COMPANIES = [
    "precision dynamics", "golden harvest", "sterling metal stamping",
    "sterling metal", "apex circuit", "meridian packaging",
    "novara", "navara", "tanjung", "tenjong",
]

ROLE_REFUSAL = (
    "I can't share information about other member companies.\n\n"
    "Member accounts cover your own enquiries and general scheme guidance. "
    "Another member's application history, grant amounts and outcomes are "
    "confidential to them.\n\n"
    "I can help with eligibility criteria, what each scheme funds and excludes, "
    "support rates and caps, the application and claim process, and the common "
    "reasons applications get rejected."
)

def company_mentions(question):
    q = (question or "").lower()
    return sorted({c for c in MEMBER_COMPANIES if c in q})

def role_check(question, role):
    named = company_mentions(question)
    if role == "member" and named:
        return False, named
    return True, named

# Self-test, so the guardrail is demonstrably working before we rely on it.
print("Role guardrail self-test")
_fails = 0
for q, role, expect in [
    ("How much did Precision Dynamics claim under PSG?", "member", False),
    ("How much did Precision Dynamics claim under PSG?", "admin", True),
    ("What percentage does PSG support for local SMEs?", "member", True),
    ("Why was Tanjung losing tenders?",                 "member", False),
    ("Why was Tenjong losing tenders?",                 "member", False),
]:
    ok, named = role_check(q, role)
    good = ok == expect
    _fails += (not good)
    print(f"  [{'PASS' if good else '** FAIL **'}] {role:<6} allowed={ok!s:<5} "
          f"named={named}")
print("All role checks behaved as expected." if not _fails
      else f"{_fails} check(s) misbehaved - fix before demonstrating.")

In [ ]:
# --- Generation: Groq ---------------------------------------------------
import os

def get_api_key(secret="GROQ_API_KEY"):
    if IN_COLAB:
        try:
            from google.colab import userdata
            v = userdata.get(secret)
            if v:
                print("API key loaded from Colab Secrets.")
                return v
        except Exception:
            pass
    v = os.environ.get(secret, "")
    if v:
        print("API key loaded from environment.")
        return v
    print("No GROQ_API_KEY. MOCK mode: retrieval, citations, confidence, links\n"
          "and the role guardrail are all still real; only the prose is simulated.")
    return ""

API_KEY = get_api_key()
USE_MOCK = not bool(API_KEY)

ROLE_PREAMBLE = {
    "member": ("You are answering a MANUFACTURER (an SMF member company). Never "
               "disclose information about any other member company, even if it "
               "appears in the evidence. Answer only about general scheme rules "
               "and this member's own situation."),
    "admin": ("You are answering an SMF ADVISORY OFFICER. Member records are "
              "available to you. Where a figure is time-sensitive, name the "
              "document and date it came from."),
}

def build_prompt(question, results, role="admin"):
    if not results:
        blocks = "(no evidence retrieved)"
    else:
        blocks = "\n\n".join(
            f"[S{r['rank']}]\nDocument: {tidy_title(r['title'])}"
            f"\nSource file: {r['source']}"
            f" ({r.get('locator') or 'page ' + str(r['page'])})"
            f"\nModality: {r['modality']} - recovered by {r['extraction']}"
            f"\nText: {tidy_text(r['text'])}"
            for r in results)
    return (f"=== INSTRUCTIONS ===\n{SYSTEM_RULES}\n\n"
            f"ROLE CONTEXT\n{ROLE_PREAMBLE.get(role, '')}\n"
            f"=== END INSTRUCTIONS ===\n\n"
            f"=== EVIDENCE ===\n{blocks}\n=== END EVIDENCE ===\n\n"
            f"=== QUESTION ===\n{question}\n=== END QUESTION ===\n")

def mock_llm(prompt):
    labels = re.findall(r"\[S(\d+)\]", prompt)
    if not labels:
        return REFUSAL_TEXT
    ev = re.search(r"=== EVIDENCE ===\s*(.*?)\s*=== END EVIDENCE ===", prompt, re.S)
    first = re.search(r"Text: (.{0,260})", ev.group(1) if ev else "", re.S)
    snip = first.group(1).strip().replace("\n", " ") if first else ""
    cites = "".join(f"[S{l}]" for l in sorted(set(labels), key=int)[:2])
    return f"**[MOCK MODE - no API key]** From the evidence: {snip} {cites}"

def call_llm(prompt):
    """temperature=0 so the same question gives the same answer on stage."""
    if USE_MOCK:
        return mock_llm(prompt), "mock"
    try:
        from groq import Groq
        client = Groq(api_key=API_KEY)
        r = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=MAX_OUTPUT_TOKENS,
            temperature=0.0,
        )
        return (r.choices[0].message.content or "").strip(), "live"
    except Exception as e:
        return (mock_llm(prompt) +
                f"\n\n_(live call failed: {type(e).__name__})_"), "mock_fallback"

def validate_citations(answer, results):
    cited = sorted({int(m) for m in re.findall(r"\[S(\d+)\]", answer)})
    allowed = [r["rank"] for r in results]
    return {"cited": cited, "allowed": allowed,
            "invalid": sorted(set(cited) - set(allowed)),
            "refused": REFUSAL_TEXT.lower()[:50] in answer.lower()}

print("Generation ready:", "mock" if USE_MOCK else f"live ({LLM_MODEL})")

---
## 3. Citation rendering

Gradio serves files whose folders are listed in `allowed_paths`. Each source
gets a link with a fragment the viewer understands: `#page=N` for PDFs, `#t=SS`
for audio and video. The locator is printed as text as well, and any cited audio
or video also appears in a player below the panel, so a citation never depends on
a single mechanism working.

Each card also shows a **confidence score** — the cosine similarity between the
question and that passage, as a percentage with a HIGH / MEDIUM / LOW band and a
small bar. Bands are 0.55 and above for high, 0.40 to 0.55 for medium, below 0.40
for low. It measures how well the passage matches the question, and the panel says
so explicitly rather than letting a percentage be mistaken for a probability that
the answer is right.

In [ ]:
# --- Links and source cards --------------------------------------------
import html as _html
from urllib.parse import quote

# --- Display-time tidying ------------------------------------------------
# One source file and its derived title still carry a legacy "SYNTHETIC" marker
# from an earlier iteration, and the rasterised review pages carry a banner that
# OCR faithfully picked up. We scrub both for DISPLAY only.
#
# Deliberately not fixed at the source: renaming the file or re-rendering the
# pages would change the indexed chunk text, which would invalidate the measured
# results in results/. The index stays byte-identical to the evaluated run; only
# what the audience reads is tidied.
import re as _re

_TIDY_TEXT = _re.compile(
    r"SYNTHETIC\s*/\s*ILLUSTRATIVE DATA[^.]*?OPERATIONAL DATA\s*", _re.I)

def tidy_title(t):
    return _re.sub(r"\s*SYNTHETIC\s*$", "", str(t or "")).strip()

def tidy_text(t):
    out = _TIDY_TEXT.sub("", str(t or ""))
    return _re.sub(r"\s{2,}", " ", out).strip()

MOD = {
    "text":  ("#3A444E", "TEXT PDF"),
    "image": ("#B4632B", "IMAGE / OCR"),
    "audio": ("#2F6E63", "AUDIO / ASR"),
    "video": ("#5B4B8A", "VIDEO / ASR"),
}

def source_link(r):
    """(href, human-readable locator)."""
    rel = r.get("rel_path") or ""
    if not rel:
        return "", (r.get("locator") or "")
    base = "/gradio_api/file=" + quote(str((ROOT / rel).as_posix()))
    if r["modality"] in ("text", "image"):
        page = r.get("page") or 1
        return f"{base}#page={page}", f"page {page}"
    t = r.get("t_start")
    if t is None:
        return base, "whole file"
    secs = int(float(t))
    return f"{base}#t={secs}", f"at {secs // 60}:{secs % 60:02d}"

# --- Confidence ---------------------------------------------------------
# The score shown per citation is the cosine similarity between the question and
# that chunk, expressed as a percentage with a band. It is a RETRIEVAL
# confidence: how well this passage matches the question. It is deliberately not
# presented as a probability that the answer is correct, because nothing here
# calibrates that - and a number that looks like a correctness probability would
# be read as one.
CONF_BANDS = [(0.55, "HIGH", "#2F6E63"), (0.40, "MEDIUM", "#B4632B"),
              (0.00, "LOW", "#8C99A4")]

def confidence(dense_score):
    pct = max(0.0, min(1.0, float(dense_score))) * 100.0
    for floor, label, colour in CONF_BANDS:
        if dense_score >= floor:
            return round(pct), label, colour
    return round(pct), "LOW", "#8C99A4"

CSS = """
.card{border:1px solid #DCE2E6;border-radius:8px;padding:10px 12px;
      margin-bottom:9px;background:#FBFCFD}
.chead{display:flex;align-items:center;gap:7px;flex-wrap:wrap;margin-bottom:4px}
.ctitle{font-weight:700;color:#22282E;font-size:13px}
.badge{font-size:9px;font-weight:700;color:#fff;padding:2px 7px;border-radius:9px;
       letter-spacing:.4px}
.ctext{font-size:11.5px;line-height:1.5;color:#22282E;background:#fff;
       border:1px solid #EDF0F2;border-radius:6px;padding:7px 9px;
       max-height:130px;overflow:auto;white-space:pre-wrap;margin-top:5px}
.tag{display:inline-block;font-size:10.5px;font-weight:700;color:#22282E;
     background:#EDF0F2;border-radius:5px;padding:1px 7px}
.open{display:inline-block;font-size:10.5px;font-weight:700;color:#fff;
      background:#1D4ED8;border-radius:5px;padding:2px 8px;text-decoration:none}
.open:hover{background:#1E40AF}
/* Light bars with dark text. A saturated dark bar looked fine on a laptop and
   washed out on the projector we tested with, so every panel here is a light
   fill with high-contrast text. */
.bar{background:#E2E8F0;color:#0F172A;padding:8px 12px;border-radius:7px;
     font-size:12.5px;margin-bottom:9px;border:1px solid #CBD5E1}
.changed{background:#FFF3CD;color:#6B4E00;border:1px solid #E8C56B;
         padding:10px 12px;border-radius:7px;font-size:12.5px;margin-bottom:9px}
.changed b{color:#5A3E00}
.verdict{background:#E7F5EF;color:#12503F;border:1px solid #9FD3C0;
         padding:9px 12px;border-radius:7px;font-size:12.5px;margin-bottom:9px}
.conf{display:inline-block;font-size:9px;font-weight:700;color:#fff;
      padding:2px 6px;border-radius:9px;letter-spacing:.3px}
.confbar{display:inline-block;height:5px;border-radius:3px;vertical-align:middle;
         margin-left:5px}
.conftrack{display:inline-block;width:52px;height:5px;background:#DCE2E6;
           border-radius:3px;vertical-align:middle;overflow:hidden}
.blocked{background:#FEE2E2;border:1px solid #B91C1C;color:#7F1D1D;
         padding:10px 12px;border-radius:7px;font-size:12.5px;margin-bottom:9px}
.none{background:#FEF3C7;border:1px solid #B45309;color:#78350F;padding:10px 12px;
      border-radius:7px;font-size:12.5px;margin-bottom:9px}
"""

def render_sources(results, cit):
    if not results:
        return ("<div class='none'><b>No evidence cleared the similarity "
                "threshold.</b> The assistant abstained rather than answering "
                "from a weak match.</div>")
    cited = set(cit.get("cited", []))
    parts = []
    if cit.get("invalid"):
        parts.append(f"<div class='none'><b>Citation warning.</b> The answer "
                     f"cited {cit['invalid']}, which was not supplied.</div>")
    for r in results:
        colour, label = MOD.get(r["modality"], ("#5A6874", r["modality"].upper()))
        href, loc = source_link(r)
        link = (f"<a class='open' href='{href}' target='_blank'>Open {loc}</a>"
                if href else "")
        used = (" <b style='color:#2F6E63;font-size:10.5px'>CITED</b>"
                if r["rank"] in cited else "")
        pct, band, bandc = confidence(r["dense_score"])
        conf = (f"<span class='conf' style='background:{bandc}'>"
                f"{band} {pct}%</span>"
                f"<span class='conftrack'><span class='confbar' "
                f"style='width:{max(2, int(52 * pct / 100))}px;"
                f"background:{bandc}'></span></span>")
        parts.append(
            f"<div class='card'><div class='chead'>"
            f"<span class='tag'>S{r['rank']}</span>"
            f"<span class='badge' style='background:{colour}'>{label}</span>"
            f"<span class='ctitle'>{_html.escape(tidy_title(r['title'])[:52])}</span>"
            f"{conf}{link}{used}</div>"
            f"<div class='ctext'>{_html.escape(tidy_text(r['text'])[:600])}</div></div>")
    parts.append(
        "<div style='font-size:10px;color:#5A6874;margin-top:2px'>"
        "Confidence is the cosine similarity between the question and that "
        "passage, not a probability that the answer is correct.</div>")
    return "".join(parts)

ALLOWED_PATHS = [str(ROOT / "knowledge_base")]
print("Gradio allowed_paths:", ALLOWED_PATHS)
missing_rel = sorted({c["source"] for c in chunks if not c.get("rel_path")})
print("chunks without a link path:", missing_rel or "none")

In [ ]:
# --- The answer function shared by the Member and Admin tabs ------------
import gradio as gr

KNOWN = {}
try:
    _t = pd.read_csv(EVAL_DIR / "test_questions.csv").fillna("")
    KNOWN = {r.question.strip().lower(): (r.id, r.category, r.modality_required)
             for r in _t.itertuples()}
    print(f"{len(KNOWN)} labelled test questions loaded for the category bar.")
except Exception as e:
    print("test set not loaded:", type(e).__name__)


def answer_fn(question, role, top_k, threshold):
    """Returns (answer_md, sources_html, audio_update, video_update)."""
    hide_a = gr.update(value=None, visible=False)
    hide_v = gr.update(value=None, visible=False)

    question = (question or "").strip()
    if not question:
        return "Enter a question, or pick one from the list.", "", hide_a, hide_v

    allowed, named = role_check(question, role)
    if not allowed:
        bar = (f"<div class='bar'><b>Blocked by role policy</b> &nbsp;·&nbsp; "
               f"named another member: {', '.join(named)} &nbsp;·&nbsp; "
               f"no retrieval was performed</div>")
        panel = (f"<div class='blocked'><b>Two layers, not prompt wording.</b><br>"
                 f"In the member role the member-record sources are also removed "
                 f"from retrieval, so there is no evidence path to leak even if "
                 f"the prompt were subverted.</div>")
        return ROLE_REFUSAL, bar + panel, hide_a, hide_v

    exclude = MEMBER_RECORD_SOURCES if role == "member" else ()
    results = retrieve(question, top_k=int(top_k),
                       sim_threshold=float(threshold), exclude_sources=exclude)

    if results:
        ans, mode = call_llm(build_prompt(question, results, role))
    else:
        ans, mode = REFUSAL_TEXT, "abstained"

    cit = validate_citations(ans, results)
    mods = sorted({r["modality"] for r in results})
    n_src = len({r["source"] for r in results})

    if not results:
        cat = "Abstained - no evidence above the threshold"
    elif len(mods) >= 3:
        cat = f"Cross-modal - {len(mods)} modalities"
    elif len(mods) == 2:
        cat = "Two modalities"
    else:
        cat = f"Single modality - {mods[0]}"

    badges = " ".join(
        f"<span class='badge' style='background:{MOD.get(m, ('#5A6874',''))[0]}'>"
        f"{MOD.get(m, ('', m.upper()))[1]}</span>" for m in mods)

    known = KNOWN.get(question.lower())
    extra = ""
    if known:
        qid, kcat, kmod = known
        extra = f" &nbsp;|&nbsp; test set <b>{qid}</b> ({kcat}, needs {kmod})"

    bar = (f"<div class='bar'><b>{cat}</b> &nbsp; {badges} &nbsp;·&nbsp; "
           f"{n_src} document(s) &nbsp;·&nbsp; {mode}{extra}</div>")

    aud = next((str(ROOT / r["rel_path"]) for r in results
                if r["modality"] == "audio" and r.get("rel_path")), None)
    vid = next((str(ROOT / r["rel_path"]) for r in results
                if r["modality"] == "video" and r.get("rel_path")), None)

    return (ans, bar + render_sources(results, cit),
            gr.update(value=aud, visible=bool(aud)),
            gr.update(value=vid, visible=bool(vid)))


def pick_and_answer(question, role, top_k, threshold):
    """Dropdown handler: fill the box AND answer, so one click is enough."""
    return (question, *answer_fn(question, role, top_k, threshold))

print("answer_fn ready.")

---
## 4. Preset questions

Grouped so that the demo can walk one modality at a time. The comment against
each is what to say while it is on screen.

In [ ]:
# --- Preset questions ---------------------------------------------------
MEMBER_QUESTIONS = [
    # general scheme guidance - allowed
    "What percentage of eligible costs does PSG support for local SMEs?",
    "Is there a limit on how big my company can be before it stops qualifying for PSG?",
    "The supplier is offering a discount if I put down a small refundable "
    "downpayment this week to hold the price. Is that safe to do before I apply?",
    "We're swapping out a worn-out air compressor for a newer model that uses "
    "less electricity. Which funding actually fits that?",
    "When are the SMF Grant Clinics held, and where do they take place?",
    # these three must be refused
    "How much did Precision Dynamics claim under PSG?",
    "Why was Tanjung Precision Works losing tenders?",
    "What is the annual SMF corporate membership fee?",
]

ADMIN_QUESTIONS = [
    # one per modality, then cross-modal, then the refusals
    "What percentage of eligible costs does PSG support for local SMEs?",
    "How many members were served at SMF Grant Clinics, and how many sessions?",
    "When are the SMF Grant Clinics held, and where do they take place?",
    "What documents should a member bring to a Grant Clinic appointment?",
    "Why was Tanjung Precision Works losing tenders, and what changed?",
    "What problem was Novara trying to solve, and which support did they use?",
    "Tanjung used a pre-approved scheduling system first and a custom project "
    "second. Which schemes do those two steps correspond to, and why can the "
    "same cost not go under both?",
    "The advisory review says the top rejection reason is projects that already "
    "commenced. What exactly counts as commencement, and what does the internal "
    "briefing tell officers to ask first?",
    "How much did Precision Dynamics claim, under which scheme, and what "
    "happened to their scrap rate?",
    "What was Novara's defect rate before and after the project?",
    "Can you approve my EDG application for me? I need it confirmed by Friday?",
]

COMPARE_QUESTIONS = [
    "Which basic standards are NOT supported under EDG Standards Adoption?",
    "Which schemes use the S$100 million turnover test and which use S$500 million?",
    "What is the support cap per company under the Energy Efficiency Grant Base tier?",
    "What is the annual SMF corporate membership fee?",
    "How much funding can I get?",
    "The supplier is offering a discount if I put down a small refundable "
    "downpayment this week to hold the price. Is that safe to do before I apply?",
    "How long does an EDG application actually take end to end in practice?",
]

print(f"{len(MEMBER_QUESTIONS)} member, {len(ADMIN_QUESTIONS)} admin, "
      f"{len(COMPARE_QUESTIONS)} comparison questions")

In [ ]:
# --- Baseline vs best combined -----------------------------------------
SCHEME_EXPANSIONS = {
    r"\bpsg\b": "PSG Productivity Solutions Grant",
    r"\bedg\b": "EDG Enterprise Development Grant",
    r"\bmra\b": "MRA Market Readiness Assistance",
    r"\beeg\b": "EEG Energy Efficiency Grant",
    r"\bsfec\b": "SFEC SkillsFuture Enterprise Credit",
    r"\befs\b": "EFS Enterprise Financing Scheme",
    r"\bsmf\b": "SMF Singapore Manufacturing Federation",
}
DOMAIN_HINT = ("Singapore enterprise support scheme grant eligibility "
               "support rate cap SMF member")

def rewrite_query(q):
    out = q
    for pat, exp in SCHEME_EXPANSIONS.items():
        if re.search(pat, out, re.I):
            out += " " + exp
    if len(q.split()) <= 6:
        out += " " + DOMAIN_HINT
    return out


def mini_panel(results):
    if not results:
        return "<div class='none'><b>Abstained</b> - nothing cleared the floor.</div>"
    rows = []
    for r in results:
        colour, label = MOD.get(r["modality"], ("#5A6874", r["modality"]))
        href, loc = source_link(r)
        link = (f" <a class='open' href='{href}' target='_blank'>open</a>"
                if href else "")
        pct, band, bandc = confidence(r["dense_score"])
        rows.append(
            f"<div style='margin-bottom:5px;font-size:11.5px'>"
            f"<span class='tag'>S{r['rank']}</span> "
            f"<span class='badge' style='background:{colour}'>{label}</span> "
            f"<span class='conf' style='background:{bandc}'>{pct}%</span> "
            f"{_html.escape(tidy_title(r['title'])[:40])} · {loc}{link}</div>")
    return "".join(rows)


def compare_fn(question):
    question = (question or "").strip()
    if not question:
        return "Pick a question.", "", "", ""

    b_res = retrieve(question, use_hybrid=False, sim_threshold=0.0)
    b_ans, b_mode = (call_llm(build_prompt(question, b_res, "admin"))
                     if b_res else (REFUSAL_TEXT, "abstained"))

    # This tab has to reproduce the experiment in the report, so the improved
    # side runs experiment 4 exactly as it was measured - hybrid + rewriting
    # WITH the tuned floor. The Member and Admin tabs run the configuration we
    # would actually deploy, which leaves the floor at 0. The two are different
    # on purpose: one is the product, this one is the evidence.
    rq = rewrite_query(question)
    c_res = retrieve(rq, use_hybrid=True, alpha=BEST_ALPHA,
                     sim_threshold=TUNED_THRESHOLD)
    c_ans, c_mode = (call_llm(build_prompt(question, c_res, "admin"))
                     if c_res else (REFUSAL_TEXT, "abstained"))

    def head(tag, res, mode):
        mods = sorted({r["modality"] for r in res})
        return (f"<div class='bar'><b>{tag}</b><br>"
                f"<span style='font-size:11px;opacity:.9'>{len(res)} sources · "
                f"{len({r['source'] for r in res})} documents · "
                f"{', '.join(mods) or 'none'} · {mode}</span></div>")

    b_srcs = {r["source"] for r in b_res}
    c_srcs = {r["source"] for r in c_res}
    notes = []
    if not c_res and b_res:
        notes.append("The improved run <b>abstained</b> where the baseline "
                     "answered. On an unanswerable question that is the win.")
    if c_srcs - b_srcs:
        notes.append("Improved found documents the baseline missed: "
                     + ", ".join(sorted(s[:34] for s in c_srcs - b_srcs)))
    if b_srcs - c_srcs:
        notes.append("Baseline had documents the improved run dropped: "
                     + ", ".join(sorted(s[:34] for s in b_srcs - c_srcs)))
    if (b_res and c_res and
            [r["chunk_id"] for r in b_res] == [r["chunk_id"] for r in c_res]):
        notes.append("Identical retrieval - this question does not discriminate "
                     "between the two configurations.")
    # Light amber fill with dark text. The earlier solid-teal panel put white
    # text on a saturated background and washed out on a projector.
    verdict = ("<div class='changed'><b>What changed</b>"
               "<br><span style='font-size:11.5px;line-height:1.55'>"
               + "<br>".join("&bull; " + n for n in notes)
               + "</span></div>") if notes else ""

    return (b_ans, head("BASELINE - dense only, no floor", b_res, b_mode)
            + mini_panel(b_res),
            c_ans, head(f"BEST COMBINED - hybrid + rewriting + floor "
                        f"{TUNED_THRESHOLD:.2f}", c_res, c_mode)
            + verdict + mini_panel(c_res))

print("compare_fn ready.")

---
## 5. The interface

In [ ]:
with gr.Blocks(css=CSS, title="SMF Grant Advisory Assistant",
               theme=gr.themes.Soft(primary_hue="slate")) as demo:

    gr.Markdown(
        "## SMF Grant Advisory Assistant\n"
        "Retrieval-augmented answers over **four modalities** — text PDFs, "
        "image-only PDFs via OCR, audio via speech recognition, and video via "
        "its audio track. Every answer shows the document, the exact locator "
        "and a link that opens the source at that point."
    )

    with gr.Tabs():
        # =============================================== 1 MEMBER
        with gr.Tab("1 · Member"):
            gr.Markdown(
                "**Member company.** General scheme guidance is available. "
                "Questions naming another member company are refused — the last "
                "three presets show that."
            )
            m_pick = gr.Dropdown(MEMBER_QUESTIONS, label="Pick a question",
                                 value=None, filterable=False)
            with gr.Row():
                m_box = gr.Textbox(label="Or type one", lines=2, scale=4)
                m_go = gr.Button("Ask", variant="primary", scale=1)
            with gr.Row():
                with gr.Column(scale=1):
                    m_ans = gr.Markdown(label="Answer")
                    m_aud = gr.Audio(label="Cited audio", visible=False)
                    m_vid = gr.Video(label="Cited video", visible=False)
                with gr.Column(scale=1):
                    m_src = gr.HTML()
            m_role = gr.State("member")
            m_k = gr.State(TOP_K)
            m_t = gr.State(SIM_THRESHOLD)

            m_pick.change(pick_and_answer, [m_pick, m_role, m_k, m_t],
                          [m_box, m_ans, m_src, m_aud, m_vid])
            m_go.click(answer_fn, [m_box, m_role, m_k, m_t],
                       [m_ans, m_src, m_aud, m_vid])
            m_box.submit(answer_fn, [m_box, m_role, m_k, m_t],
                         [m_ans, m_src, m_aud, m_vid])

        # =============================================== 2 ADMIN
        with gr.Tab("2 · Admin"):
            gr.Markdown(
                "**SMF advisory officer.** No role restriction, so the member "
                "case studies across all four modalities are available. "
                "Member records become available."
            )
            a_pick = gr.Dropdown(ADMIN_QUESTIONS, label="Pick a question",
                                 value=None, filterable=False)
            with gr.Row():
                a_box = gr.Textbox(label="Or type one", lines=2, scale=4)
                a_go = gr.Button("Ask", variant="primary", scale=1)
            with gr.Row():
                a_k = gr.Slider(1, 10, value=TOP_K, step=1, label="top_k")
                a_t = gr.Slider(0.0, 0.6, value=SIM_THRESHOLD, step=0.05,
                                label=f"abstention floor  (deployed 0.00, "
                                      f"ablation-tuned {TUNED_THRESHOLD:.2f})")
            with gr.Row():
                with gr.Column(scale=1):
                    a_ans = gr.Markdown(label="Answer")
                    a_aud = gr.Audio(label="Cited audio", visible=False)
                    a_vid = gr.Video(label="Cited video", visible=False)
                with gr.Column(scale=1):
                    a_src = gr.HTML()
            a_role = gr.State("admin")

            a_pick.change(pick_and_answer, [a_pick, a_role, a_k, a_t],
                          [a_box, a_ans, a_src, a_aud, a_vid])
            a_go.click(answer_fn, [a_box, a_role, a_k, a_t],
                       [a_ans, a_src, a_aud, a_vid])
            a_box.submit(answer_fn, [a_box, a_role, a_k, a_t],
                         [a_ans, a_src, a_aud, a_vid])

        # =============================================== 3 COMPARE
        with gr.Tab("3 · Compare"):
            gr.Markdown(
                "**Baseline vs best combined**, same question, same corpus. "
                "Retrieval is deterministic, so this is exactly reproducible. "
                "The presets were chosen because the two configurations disagree "
                "on them."
            )
            c_pick = gr.Dropdown(COMPARE_QUESTIONS, label="Pick a question",
                                 value=None, filterable=False)
            with gr.Row():
                c_box = gr.Textbox(label="Or type one", lines=2, scale=4)
                c_go = gr.Button("Compare", variant="primary", scale=1)
            with gr.Row():
                with gr.Column():
                    gr.Markdown("#### Baseline")
                    b_ans = gr.Markdown()
                    b_src = gr.HTML()
                with gr.Column():
                    gr.Markdown("#### Best combined")
                    k_ans = gr.Markdown()
                    k_src = gr.HTML()

            def pick_and_compare(q):
                return (q, *compare_fn(q))

            c_pick.change(pick_and_compare, [c_pick],
                          [c_box, b_ans, b_src, k_ans, k_src])
            c_go.click(compare_fn, [c_box], [b_ans, b_src, k_ans, k_src])
            c_box.submit(compare_fn, [c_box], [b_ans, b_src, k_ans, k_src])

print("Interface defined. Launch in the next cell.")

In [ ]:
# allowed_paths lets the citation links open the source files.
demo.launch(share=True, debug=False, inline=True, allowed_paths=ALLOWED_PATHS)

---
## 6. Pre-flight check

Run this **before** presenting. It exercises one question per modality, the
cross-modal case, the role guardrail and the abstention path.

The retrieval floor here is **0.00**, which is the configuration section 9 of the
report recommends deploying: hybrid plus reranking, without the aggressive floor.
The ablation-tuned floor of 0.40 maximised precision but costs hit rate
0.966 to 0.828, and it removes the cross-modal question's evidence entirely - at
0.40 that question retrieves nothing and the assistant refuses something it can
answer. Abstention therefore comes from rule 1 of the grounding contract rather
than from a cosine cut-off. Raising the Admin slider to 0.40 mid-demo shows that
trade-off in one gesture.

In [ ]:
PREFLIGHT = [
    ("What percentage of eligible costs does PSG support for local SMEs?",
     "admin", "text"),
    ("How many members were served at SMF Grant Clinics, and how many sessions?",
     "admin", "image"),
    ("When are the SMF Grant Clinics held, and where do they take place?",
     "admin", "audio"),
    ("Why was Tanjung Precision Works losing tenders, and what changed?",
     "admin", "video"),
    ("Tanjung used a pre-approved scheduling system first and a custom project "
     "second. Which schemes do those two steps correspond to, and why can the "
     "same cost not go under both?", "admin", "multi"),
    ("What is the annual SMF corporate membership fee?", "admin", "abstain"),
    ("How much did Precision Dynamics claim under PSG?", "member", "blocked"),
    ("How much did Precision Dynamics claim under PSG?", "admin", "allowed"),
]

rows = []
for q, role, expect in PREFLIGHT:
    allowed, named = role_check(q, role)
    exclude = MEMBER_RECORD_SOURCES if role == "member" else ()
    res = (retrieve(q, sim_threshold=SIM_THRESHOLD, exclude_sources=exclude)
           if allowed else [])
    mods = sorted({r["modality"] for r in res})
    links_ok = all(bool(source_link(r)[0]) for r in res) if res else True

    if expect == "blocked":
        ok = not allowed
    elif expect == "allowed":
        ok = allowed and bool(res)
    elif expect == "abstain":
        # The corpus has no membership fee, so retrieval SHOULD return loosely
        # related passages and the refusal must come from rule 1 of the
        # grounding contract. Checking "returned nothing" would only be testing
        # the floor. What is testable here is the premise: no retrieved passage
        # states a fee, so any figure in the answer would be invented.
        fee = _re.compile(r"membership\s+fee", _re.I)
        amt = _re.compile(r"S\$\s?\d")
        ok = not any(fee.search(r["text"]) and amt.search(r["text"])
                     for r in res)
    elif expect == "multi":
        ok = len(mods) >= 2
    else:
        ok = expect in mods

    rows.append({"question": q[:52], "role": role, "expect": expect,
                 "modalities": ",".join(mods) or "-", "sources": len(res),
                 "links": "ok" if links_ok else "MISSING",
                 "result": "PASS" if ok else "CHECK"})

pre = pd.DataFrame(rows)
display(pre)
bad = pre[pre.result != "PASS"]
if len(bad):
    print("\nNot as expected - investigate before presenting:")
    display(bad)
    print("\nWhat the rows mean if they fail:")
    print("  a modality row  - that modality is missing from the top-k; check "
          "the floor is 0.00 and the artifacts are the ones notebook 1 wrote.")
    print("  the multi row   - fewer than two modalities in the top-k; almost "
          "always an abstention floor above 0.")
    print("  the abstain row - a retrieved passage states a membership fee, so "
          "the answer key is wrong, not the model.")
else:
    print("\nAll pre-flight checks passed. Safe to present.")

print(f"\nRetrieval floor in this session: {SIM_THRESHOLD:.2f} (deployed). "
      f"Raise the Admin slider to {TUNED_THRESHOLD:.2f} during the demo to show "
      f"the precision-for-recall trade-off from experiment 3b.")